# BirdCLEF+ 2025 - Pretrained models: seed repeats of the two distilled heads (noisy-student round 1, step 5)

**CPU only, internet off, no submission, no heads saved.** We run seeds 42, 43 and 44 for:

1. **a3_distill** - the A3 `distill` head of `a3-distill` (variant `distill`, source features untransformed): Bird-MAE MLP head (hidden 512, dropout 0.2, max_lr 3e-3, wd 1e-2, 30 epochs, batch 1024, one-cycle pct_start 0.1) on train-fold Bird-MAE windows with real labels and per-clip weights plus the 12,000 soundscape Bird-MAE windows; teacher = original A2 head `perch_head.npz` on the soundscape Perch embeddings, `p ** 1.3`, `< 0.01 -> 0`, weight 0.5 per window.
2. **a2_a1distill** - the A2 + A1-distilled head of `a2-a1distill`: Perch MLP head (hidden 512, dropout 0.3, max_lr 1e-3, same schedule) on train-fold Perch windows plus the 12,000 soundscape Perch windows; teacher = the saved `teacher_a1_probs.npy` of `a2-a1distill` v1 (no GPU), same sharpening and weight.

**The seed changes head initialisation, batch order and dropout only; the stratified 80/20 split is seed 42 for every run.** Both recipes are copied from the original `train_run`, including the selection of the best epoch by first-5s val AUC. We compare seed 42 with the saved first-5s values (a3_distill 0.97504 on Kaggle CPU; a2_a1distill 0.99182 on a T4). The CPU rerun is not expected to match it exactly, because the dropout masks come from a different generator. Std = sample std (ddof 1). Output: `head_seeds.json`.

In [ ]:
import os
N_THREADS = int(os.environ.get("M3_THREADS", os.cpu_count() or 4))
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = str(N_THREADS)         # set before numpy loads; all cores for the matrix work

import ast, glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

T0 = time.time()
torch.set_num_threads(N_THREADS)
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"     # local test only, never set on Kaggle
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
print("device: cpu | torch threads:", torch.get_num_threads(), "| os.cpu_count():", os.cpu_count())
if torch.cuda.is_available():
    print("NOTE: a GPU is visible but not used; this notebook trains on the CPU like a3-distill")

CFG = dict(val_frac=0.2, seed=42, secondary_weight=0.5, batch_size=1024, weight_decay=1e-2,
           epochs=int(os.environ.get("M3_EPOCHS", "2")) if SMOKE else 30)
HEADS = dict(
    # first: the larger one, so the projection after its seed 42 is an upper bound
    a2_a1distill=dict(dim=1536, emb_dir="perch", emb="perch_embeddings.npy", windows="perch_windows.csv",
                      target="perch_soundscape_embeddings.npy", hidden=512, dropout=0.3, max_lr=1e-3, teacher="a1",
                      saved_first5s=0.9918249582214589, saved_on="T4 GPU, a2-a1distill v1"),
    a3_distill=dict(dim=768, emb_dir="birdmae", emb="birdmae_embeddings.npy", windows="birdmae_windows.csv",
                    target="birdmae_soundscape_embeddings.npy", hidden=512, dropout=0.2, max_lr=3e-3, teacher="a2",
                    saved_first5s=0.9750438851600266, saved_on="Kaggle CPU, a3-distill v1 (variant distill)"),
)
SEEDS = (42, 43, 44)                       # 42 first: the saved-value check and the timing guard use it
SOUNDSCAPE_WEIGHT = 0.5                    # literal loss weight per soundscape window
TEACHER_POWER, TEACHER_FLOOR = 1.3, 0.01
N_SOUNDSCAPE_WINDOWS = 12000
# a3-distill v1 teacher statistics (distill_results.json), to confirm the A2 teacher is rebuilt identically
A2_TEACHER_SAVED = dict(fraction_nonzero=0.018767394822006473, windows_any_label_ge_0_5=1431)
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 45 * 60))          # env override: local guard test only
SAFETY = 1.25
STRICT = set(filter(None, os.environ.get("M3_STRICT", "").split(",")))
def strict(name):
    return not SMOKE or name in STRICT

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {marker} under /kaggle/input, found {hits}")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
SOUND = find_dir("soundscape_embed_meta.json", "M3_SOUNDSCAPE_DIR")
PERCH_DIR = find_dir("perch_embeddings.npy", "M3_PERCH_DIR")
A2_HEAD_DIR = find_dir("perch_head.npz", "M3_A2_HEAD_DIR")          # the original A2 head, exact name
BIRDMAE_DIR = find_dir("birdmae_embeddings.npy", "M3_BIRDMAE_DIR")
A1D_DIR = find_dir("teacher_a1_probs.npy", "M3_A1D_DIR")             # a2-a1distill v1 output
DIRS = dict(perch=PERCH_DIR, birdmae=BIRDMAE_DIR)
print("Data:", DATA, "\nM1 index:", MELS, "\nsoundscapes:", SOUND, "\nperch:", PERCH_DIR, "\nA2 head:", A2_HEAD_DIR,
      "\nbirdmae:", BIRDMAE_DIR, "\na2-a1distill:", A1D_DIR)

In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va), flush=True)

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Keep identical to the inference notebooks."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

## Teacher soft labels (no GPU): A2 head on the soundscape Perch embeddings, and the saved A1 probabilities

In [ ]:
t_teacher = time.time()
windows_s = pd.read_csv(SOUND / "soundscape_windows.csv")
N_S = len(windows_s)
assert json.load(open(SOUND / "soundscape_embed_meta.json"))["windows"] == N_S
if strict("n"):
    assert N_S == N_SOUNDSCAPE_WINDOWS, f"expected {N_SOUNDSCAPE_WINDOWS} soundscape windows, got {N_S}"
for d in (PERCH_DIR, A2_HEAD_DIR, BIRDMAE_DIR, A1D_DIR):
    assert json.load(open(d / "species.json")) == species, f"{d}: species list differs from sample_submission"
assert json.load(open(A2_HEAD_DIR / "head_cfg.json"))["best_head"] == "mlp", "perch_head.npz is not the MLP head"

def sharpen(p):
    q = (p ** TEACHER_POWER).astype(np.float32)
    q[q < TEACHER_FLOOR] = 0
    assert q.dtype == np.float32 and np.isfinite(q).all() and q.shape == (N_S, len(species))
    return q

def stats(q):
    return dict(fraction_nonzero=float((q > 0).mean()), windows_any_label_ge_0_5=int((q.max(axis=1) >= 0.5).sum()),
                mean_max_per_window=float(q.max(axis=1).mean()))

def check_stats(name, got, saved):
    d_frac = abs(got["fraction_nonzero"] - saved["fraction_nonzero"])
    d_n = abs(got["windows_any_label_ge_0_5"] - saved["windows_any_label_ge_0_5"])
    print(f"[{name}] teacher fraction non-zero {got['fraction_nonzero']:.6f} (saved {saved['fraction_nonzero']:.6f}) | "
          f"windows >= 0.5 {got['windows_any_label_ge_0_5']} (saved {saved['windows_any_label_ge_0_5']})", flush=True)
    if (d_frac > 1e-4 or d_n > 2) and strict("teacher"):
        raise RuntimeError(f"{name}: teacher labels differ from the saved run (fraction diff {d_frac:.2e}, windows diff {d_n})")

# (1) A2 teacher for a3_distill: exactly as a3-distill
P_S = np.load(SOUND / "perch_soundscape_embeddings.npy").astype(np.float32)
assert P_S.shape == (N_S, 1536) and np.isfinite(P_S).all(), P_S.shape
teacher_head = dict(np.load(A2_HEAD_DIR / "perch_head.npz"))
assert int(teacher_head["n_layers"]) == 2 and teacher_head["W0"].shape[1] == 1536 and teacher_head["W1"].shape[0] == len(species)
SOFT = dict(a2=sharpen(head_predict(teacher_head, P_S).astype(np.float32)))
del P_S
TEACHER_STATS = dict(a2=stats(SOFT["a2"]))
check_stats("A2 head (a3_distill teacher)", TEACHER_STATS["a2"], A2_TEACHER_SAVED)

# (2) A1 teacher for a2_a1distill: the saved raw sigmoid probabilities of a2-a1distill v1 (csv row order)
p_a1 = np.load(A1D_DIR / "teacher_a1_probs.npy")
assert p_a1.dtype == np.float32 and p_a1.shape == (N_S, len(species)), (p_a1.dtype, p_a1.shape)
assert np.isfinite(p_a1).all() and p_a1.min() >= 0 and p_a1.max() <= 1
SOFT["a1"] = sharpen(p_a1)
del p_a1
TEACHER_STATS["a1"] = stats(SOFT["a1"])
saved_a1 = json.load(open(A1D_DIR / "a1distill_results.json"))["teacher"]
check_stats("A1 probs (a2_a1distill teacher)", TEACHER_STATS["a1"], saved_a1)
TEACHER_S = time.time() - t_teacher

## Head training (the original train_run of a3-distill / a2-a1distill, with the seed as an argument, no head saved)

In [ ]:
def make_head(cfg):
    return nn.Sequential(nn.Linear(cfg["dim"], cfg["hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(cfg["hidden"], len(species)))

def train_run(name, cfg, d, seed, X, T32, soft):
    """One full run. The seed sets head initialisation, batch order and dropout; the split stays seed 42."""
    t_start = time.time()
    torch.manual_seed(seed); np.random.seed(seed)
    mean, std = X[d.in_train].mean(axis=0), X[d.in_train].std(axis=0) + 1e-6
    # The standardiser comes from the train-fold source windows only (recipe unchanged); the soundscape windows use it too.
    n_src, n_t = len(X), len(T32)
    Z = torch.from_numpy(np.concatenate([(X - mean) / std, (T32 - mean) / std]))
    Y_all = torch.cat([d.Y_t, torch.from_numpy(soft)])
    W_all = torch.cat([d.W_t, torch.full((n_t,), SOUNDSCAPE_WEIGHT, dtype=torch.float32)])
    tr_rows = torch.from_numpy(np.concatenate([np.where(d.in_train)[0], np.arange(n_src, n_src + n_t)]))
    w_src = float(d.W_t.numpy()[d.in_train].sum()); w_sound = SOUNDSCAPE_WEIGHT * n_t
    share = w_sound / (w_src + w_sound)

    @torch.no_grad()
    def predict(model, rows):
        model.eval(); out = []
        for i in range(0, len(rows), 4096):
            out.append(torch.sigmoid(model(Z[torch.from_numpy(rows[i:i + 4096])])).numpy())
        return np.concatenate(out)

    def clip_max_auc(model):
        p = pd.DataFrame(predict(model, d.rows_all_va)).groupby(d.va_groups).max().loc[d.order_clips].values
        return macro_auc(d.Y_clip, p)[0]

    model = make_head(cfg)
    epochs = CFG["epochs"]
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["max_lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["max_lr"], total_steps=epochs * steps, pct_start=0.1)
    best = dict(first=-1.0); tag = f"{name}/seed{seed}"
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows))]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            bce = nn.functional.binary_cross_entropy_with_logits(model(Z[b]), Y_all[b], reduction="none")
            loss = (bce.mean(1) * W_all[b]).sum() / W_all[b].sum()
            opt.zero_grad(set_to_none=True)
            loss.backward(); opt.step(); sched.step()
            total += loss.item()
        first, n_cls = macro_auc(d.Y_va, predict(model, d.rows_va))
        flag = ""
        if first > best["first"]:
            best = dict(first=first, clip_max=clip_max_auc(model), best_epoch=ep, n_classes=n_cls); flag = "  <- best"
        print(f"[{tag}] ep {ep:2d}/{epochs} | loss {total / steps:.4f} | first-5s {first:.4f} | {time.time() - t0:.1f}s{flag}", flush=True)
    return dict(val_auc_first5s=float(best["first"]), val_auc_clip_max=float(best["clip_max"]), best_epoch=int(best["best_epoch"]),
                val_classes=int(best["n_classes"]), seed=seed, soundscape_loss_weight_share=share, seconds=time.time() - t_start)

## Runs

In [ ]:
class Data:
    pass

def prepare(cfg):
    t0 = time.time(); d = Data()
    cache = DIRS[cfg["emb_dir"]]
    val_clips = pd.read_csv(cache / "val_clips.csv")
    if not np.array_equal(val_clips["clip"].values, idx_va):
        raise RuntimeError(f"{cfg['emb_dir']}: val_clips.csv does not match the recomputed split")
    if "filename" in index.columns and not (index["filename"].astype(str).values[idx_va] == val_clips["filename"].astype(str).values).all():
        raise RuntimeError(f"{cfg['emb_dir']}: val_clips.csv filenames do not match train_index.csv")
    E = np.load(cache / cfg["emb"]).astype(np.float32)
    windows = pd.read_csv(cache / cfg["windows"])
    if E.shape[1] != cfg["dim"] or len(E) != len(windows) or not np.isfinite(E).all():
        raise RuntimeError(f"{cfg['emb_dir']}: bad source embeddings {E.shape} vs {len(windows)} windows")
    T32 = np.load(SOUND / cfg["target"]).astype(np.float32)      # untransformed
    if T32.shape != (N_S, cfg["dim"]) or not np.isfinite(T32).all():
        raise RuntimeError(f"{cfg['emb_dir']}: bad soundscape embeddings {T32.shape}")
    d.E, d.T32 = E, T32
    clip_of, win_of = windows["clip"].values, windows["win"].values
    d.in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
    assert not (d.in_train & in_val).any() and (d.in_train | in_val).all()
    Y = np.zeros((len(windows), len(species)), np.float32)
    for row, c in enumerate(clip_of):
        Y[row, sec_idx[c]] = CFG["secondary_weight"]
        Y[row, y_idx[c]] = 1.0
    clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
    d.W_t = torch.from_numpy((1.0 / clip_windows).astype(np.float32)); d.Y_t = torch.from_numpy(Y)
    d.rows_va = np.where(in_val & (win_of == 0))[0]
    d.Y_va = np.zeros((len(d.rows_va), len(species)), np.float32)
    d.Y_va[np.arange(len(d.rows_va)), y_idx[clip_of[d.rows_va]]] = 1.0
    d.rows_all_va = np.where(in_val)[0]
    d.va_groups = clip_of[d.rows_all_va]
    d.order_clips = np.unique(d.va_groups)
    d.Y_clip = np.zeros((len(d.order_clips), len(species)), np.float32)
    d.Y_clip[np.arange(len(d.order_clips)), y_idx[d.order_clips]] = 1.0
    print(f"[{cfg['emb_dir']}] windows: source {len(E)} | train {d.in_train.sum()} | val {in_val.sum()} | val first-5s {len(d.rows_va)} | soundscape {len(T32)}", flush=True)
    d.prep_s = time.time() - t0
    return d

def check_budget(projected, limit=BUDGET_S):
    print(f"TIMING guard | projected notebook total {projected / 60:.1f} min (limit {limit / 60:.0f})", flush=True)
    if projected > limit:
        raise RuntimeError(f"Projected {projected / 60:.1f} min exceeds the {limit / 60:.0f} min limit, stopping")

results = dict(split="fixed: stratified 80/20 with seed 42 for every run; the seed varies head initialisation, batch order and dropout only",
               seeds=list(SEEDS), std_ddof=1, device="cpu", torch_threads=N_THREADS, teacher_stats=TEACHER_STATS,
               teacher_power=TEACHER_POWER, teacher_floor=TEACHER_FLOOR, soundscape_loss_weight=SOUNDSCAPE_WEIGHT,
               epochs=CFG["epochs"], heads={})
names = list(HEADS)
for pos, name in enumerate(names):
    cfg = HEADS[name]
    d = prepare(cfg)
    entry = dict(teacher=cfg["teacher"], n_source_windows=int(len(d.E)), n_train_windows=int(d.in_train.sum()),
                 n_soundscape_windows=int(len(d.T32)), prep_seconds=d.prep_s, seeds={})
    for s_pos, seed in enumerate(SEEDS):
        r = train_run(name, cfg, d, seed, d.E, d.T32, SOFT[cfg["teacher"]])
        entry["seeds"][str(seed)] = r
        print(f"RESULT [{name}/seed{seed}] first-5s {r['val_auc_first5s']:.5f} | clip-max {r['val_auc_clip_max']:.5f} | best ep {r['best_epoch']} | {r['seconds']:.0f}s", flush=True)
        if seed == 42:
            diff = r["val_auc_first5s"] - cfg["saved_first5s"]
            entry["seed42_vs_saved"] = dict(saved_first5s=cfg["saved_first5s"], saved_on=cfg["saved_on"], seed42_first5s=r["val_auc_first5s"],
                                            diff=diff, abs_diff=abs(diff), rerun_on="Kaggle CPU")
            print(f"[{name}] seed 42 first-5s {r['val_auc_first5s']:.5f} vs saved {cfg['saved_first5s']:.5f} ({cfg['saved_on']}) | "
                  f"difference {diff:+.5f}", flush=True)
            if abs(diff) > 0.015 and strict("saved"):
                raise RuntimeError(f"{name}: seed 42 differs from the saved value by {abs(diff):.4f} (> 0.015), the recipe copy is wrong")
            runs_left = (len(SEEDS) - 1 - s_pos) + (len(names) - pos - 1) * len(SEEDS)
            check_budget(time.time() - T0 + SAFETY * runs_left * (r["seconds"] + d.prep_s / len(SEEDS)))
        results["heads"][name] = entry
        json.dump(to_py(results), open(OUT / "head_seeds.json", "w"), indent=1)
    for key in ("val_auc_first5s", "val_auc_clip_max"):
        v = np.array([s[key] for s in entry["seeds"].values()])
        entry[key.replace("val_auc_", "") + "_mean"] = float(v.mean())
        entry[key.replace("val_auc_", "") + "_std"] = float(v.std(ddof=1))
    results["heads"][name] = entry
    json.dump(to_py(results), open(OUT / "head_seeds.json", "w"), indent=1)
    del d

## Summary

In [ ]:
results["teacher_seconds"] = TEACHER_S
results["total_seconds"] = time.time() - T0
json.dump(to_py(results), open(OUT / "head_seeds.json", "w"), indent=1)
for name, e in results["heads"].items():
    runs = " | ".join(f"seed {s}: {v['val_auc_first5s']:.5f}/{v['val_auc_clip_max']:.5f}/ep{v['best_epoch']}" for s, v in e["seeds"].items())
    print(f"{name:13s} | {runs}")
    print(f"{'':13s} | first-5s {e['first5s_mean']:.5f} +/- {e['first5s_std']:.5f} | clip-max {e['clip_max_mean']:.5f} +/- {e['clip_max_std']:.5f} | "
          f"seed 42 vs saved {e['seed42_vs_saved']['diff']:+.5f}")
parts = " | ".join(f"{n} prep {e['prep_seconds']:.0f}s + runs " + "/".join(f"{v['seconds']:.0f}" for v in e["seeds"].values()) + "s"
                   for n, e in results["heads"].items())
print(f"TIMING | teacher {TEACHER_S:.0f}s | {parts} | notebook total {results['total_seconds'] / 60:.1f} min | CPU, {N_THREADS} threads")